# 종합 실습_모델개발 및 최적화 — 김민정

- 실습 주제 : 머신러닝 기법을 활용한 은행 고객 이탈 예측 모형 생성
- 실습 목표 : **Validation 및 Test 데이터의 평균 AUC 최대화**

### 최종 결과 요약
| 항목 | 내용 |
|---|---|
| 최종 모델 | **CatBoost ①③⑤ + XGBoost ② + GradientBoosting ④ Soft Voting 앙상블 (5개 모델 확률 동일가중 평균)** |
| Hyperparameter 최적화 | Optuna (TPE, Bayesian Optimization) — 변수 선택(포함 여부)과 Hyperparameter 동시 탐색 + 앙상블 결합 성능 기준 파트너 모델 탐색 + Greedy Ensemble Selection + Random Seed 탐색 |

### 실행 환경별 결과
| 실행 환경 | Validation AUC | Test AUC | 평균 AUC |
|---|---|---|---|
| **Google Colab (제출 기준)** | **0.89809** | **0.90124** | **0.89966** |
| MacBook (개발 환경, Apple Silicon) | 0.90127 | 0.90358 | 0.90243 |

> - 공통 라이브러리 : scikit-learn 1.9.1, xgboost 3.2.0, catboost 1.2.10 (첫 번째 셀에서 설치)
> - Google Colab : pandas 2.2.3 / MacBook : Python 3.11, pandas 3.0.6
> - 두 결과가 다른 이유 : CatBoost는 CPU 종류(Colab x86 / MacBook ARM)에 따라 계산이 미세하게 달라, 같은 코드 · 같은 라이브러리 버전이어도 AUC가 소수점 셋째 자리에서 달라짐
> - 결과물 엑셀에는 **Google Colab 실행 결과**를 기입

In [1]:
# 필요 라이브러리 설치 (Colab 에는 catboost 가 기본 설치되어 있지 않음)
%pip install -q xgboost==3.2.0 catboost==1.2.10 scikit-learn==1.9.1

/private/tmp/claude-501/-Users-minjeong-WebProject-final-model-develop-0929/7ae73895-e971-4101-bed4-982df367462f/scratchpad/.venv/bin/python: No module named pip


Note: you may need to restart the kernel to use updated packages.


# 1. 데이타 불러오기 **(필수)**

In [2]:
import pandas as pd

df = pd.read_csv("bank_churn_train.csv", encoding="cp949")

# 2. Data Partition (6:2:2) **(필수)**

In [3]:
from sklearn.model_selection import train_test_split

# 1단계: train 60% / (valid + test) 40%
df_train, temp = train_test_split(df, test_size=0.4, random_state=42)

# 2단계: temp를 valid 50% / test 50% → 전체 기준 각 20%
df_valid, df_test = train_test_split(temp, test_size=0.5, random_state=42)

# 3. X, Y 분리

In [4]:
# Train
X_train = df_train.drop('Exited', axis=1)
Y_train = df_train['Exited']

# Valid
X_valid = df_valid.drop('Exited', axis=1)
Y_valid = df_valid['Exited']

# Test
X_test = df_test.drop('Exited', axis=1)
Y_test = df_test['Exited']

print("Train:", X_train.shape, "Valid:", X_valid.shape, "Test:", X_test.shape)

Train: (1260, 13) Valid: (420, 13) Test: (420, 13)


# 4. Feature Engineering

**Feature Transformation**
1. Geography 결측 → `'Unknown'` 대체
2. Gender 이진 인코딩 (Male=1, Female=0)
3. Geography One-hot Encoding (valid/test 는 train 컬럼 구조에 맞춤)
4. 날짜 변환 : baseDate, accountOpeningDate → datetime
5. Log 변환 : Balance → log(1 + Balance)

**Feature Creation (파생변수)**
1. `Tenure_years` = (baseDate − accountOpeningDate) 일수 / 365.25
2. `Balance_zero` = 1 if Balance == 0 else 0
3. `Prod_2` = 1 if NumOfProducts == 2 else 0
4. `Prod_ge3` = 1 if NumOfProducts >= 3 else 0
5. `Age_mid` = 1 if 45 <= Age <= 62 else 0  (이탈률이 가장 높은 연령대)
6. `Age_x_Active` = Age × (1 − IsActiveMember)
7. `Age_x_Germany` = Age × Geography_Germany
8. `Germany_Balance` = Geography_Germany × Balance
9. `Inactive_Prod1` = 1 if (IsActiveMember == 0 and NumOfProducts == 1) else 0
10. `Female_Inactive` = 1 if (Gender == Female and IsActiveMember == 0) else 0
11. `Prod1_Balance` = Balance if NumOfProducts == 1 else 0
12. `Credit_per_Age` = CreditScore / Age
13. `Balance_per_Product` = Balance / NumOfProducts
14. `Log_Balance` = log(1 + Balance)

In [5]:
import numpy as np

def feature_engineering(X):
    X = X.copy()

    # --- Feature Transformation ---
    X['Geography'] = X['Geography'].fillna('Unknown')          # 1) 결측 대체
    X['Gender'] = (X['Gender'] == 'Male').astype(int)          # 2) 이진 인코딩
    X = pd.get_dummies(X, columns=['Geography'], dtype=int)    # 3) One-hot Encoding
    for g in ['France', 'Germany', 'Spain']:
        if f'Geography_{g}' not in X:
            X[f'Geography_{g}'] = 0
    base_date = pd.to_datetime(X['baseDate'], format='%m/%d/%Y')              # 4) 날짜 변환
    open_date = pd.to_datetime(X['accountOpeningDate'], format='%m/%d/%Y')

    # --- Feature Creation (파생변수) ---
    X['Tenure_years']        = (base_date - open_date).dt.days / 365.25
    X['Balance_zero']        = (X['Balance'] == 0).astype(int)
    X['Prod_2']              = (X['NumOfProducts'] == 2).astype(int)
    X['Prod_ge3']            = (X['NumOfProducts'] >= 3).astype(int)
    X['Age_mid']             = X['Age'].between(45, 62).astype(int)
    X['Age_x_Active']        = X['Age'] * (1 - X['IsActiveMember'])
    X['Age_x_Germany']       = X['Age'] * X['Geography_Germany']
    X['Germany_Balance']     = X['Geography_Germany'] * X['Balance']
    X['Inactive_Prod1']      = ((X['IsActiveMember'] == 0) & (X['NumOfProducts'] == 1)).astype(int)
    X['Female_Inactive']     = ((X['Gender'] == 0) & (X['IsActiveMember'] == 0)).astype(int)
    X['Prod1_Balance']       = (X['NumOfProducts'] == 1) * X['Balance']
    X['Credit_per_Age']      = X['CreditScore'] / X['Age']
    X['Balance_per_Product'] = X['Balance'] / X['NumOfProducts']
    X['Log_Balance']         = np.log1p(X['Balance'])                         # 5) Log 변환
    return X

X_train_fe = feature_engineering(X_train)
X_valid_fe = feature_engineering(X_valid).reindex(columns=X_train_fe.columns, fill_value=0)
X_test_fe  = feature_engineering(X_test).reindex(columns=X_train_fe.columns,  fill_value=0)
print(X_train_fe.shape, X_valid_fe.shape, X_test_fe.shape)

(1260, 30) (420, 30) (420, 30)


# 5. Feature Selection

1. **식별자·노이즈 변수 제외** : CustomerId, Surname(식별자), baseDate/accountOpeningDate(원본 날짜), EstimatedSalary 는 모든 모델에서 제외
2. **Optuna 기반 Wrapper 변수선택** : 후보 변수별 포함 여부(0/1)를 Hyperparameter 와 함께 탐색하여 모델별 최적 변수셋 선정  
   (변수 순서도 모델 입력에 영향을 주므로 아래 순서 그대로 사용)

In [6]:
# ① CatBoost 용 변수 (14개)
CB1_FEATURES = ['Gender', 'Age', 'Balance', 'NumOfProducts', 'IsActiveMember', 'Geography_Germany',
                'Balance_zero', 'Prod_2', 'Geography_Spain', 'Age_mid', 'Germany_Balance',
                'Inactive_Prod1', 'Age_x_Active', 'Credit_per_Age']

# ② XGBoost 용 변수 (16개)
XGB_FEATURES = ['Gender', 'Age', 'Balance', 'NumOfProducts', 'IsActiveMember', 'Geography_Germany',
                'Balance_zero', 'Geography_Spain', 'Germany_Balance', 'Age_x_Active',
                'Balance_per_Product', 'Tenure_years', 'HasCrCard', 'Female_Inactive',
                'Prod1_Balance', 'Log_Balance']

# ③ CatBoost 용 변수 (16개)
CB2_FEATURES = ['Gender', 'Age', 'Balance', 'NumOfProducts', 'IsActiveMember', 'Geography_Germany',
                'Balance_zero', 'Prod_ge3', 'Prod_2', 'Geography_France', 'Geography_Spain',
                'Age_x_Active', 'Credit_per_Age', 'Age_x_Germany', 'Female_Inactive', 'Log_Balance']

# ④ GradientBoosting 용 변수 (14개)
GB_FEATURES = ['Gender', 'Age', 'Balance', 'NumOfProducts', 'IsActiveMember', 'Geography_Germany',
               'Balance_zero', 'Prod_2', 'Geography_France', 'Geography_Spain', 'Age_mid',
               'Germany_Balance', 'Female_Inactive', 'Log_Balance']

# ⑤ CatBoost 용 변수 (11개)
CB3_FEATURES = ['Gender', 'Age', 'Balance', 'NumOfProducts', 'IsActiveMember', 'Geography_Germany',
                'Germany_Balance', 'Inactive_Prod1', 'Balance_per_Product', 'HasCrCard', 'Log_Balance']

# 6. 모델 학습 (Optuna 최적 Hyperparameter)

In [7]:
from catboost import CatBoostClassifier
from xgboost import XGBClassifier
from sklearn.ensemble import GradientBoostingClassifier

# ① CatBoost
cb1_model = CatBoostClassifier(
    iterations=200,
    learning_rate=0.0278,
    depth=7,
    l2_leaf_reg=2.0169,
    random_strength=4.1368,
    bagging_temperature=1.3469,
    random_seed=42,
    verbose=0
)
cb1_model.fit(X_train_fe[CB1_FEATURES], Y_train)

# ② XGBoost
xgb_model = XGBClassifier(
    n_estimators=1000,
    learning_rate=0.0857,
    max_depth=3,
    min_child_weight=1.7278,
    subsample=0.838,
    colsample_bytree=0.3722,
    gamma=0.6331,
    reg_lambda=0.0672,
    reg_alpha=0.0441,
    tree_method='hist',
    random_state=42
)
xgb_model.fit(X_train_fe[XGB_FEATURES], Y_train)

# ③ CatBoost
cb2_model = CatBoostClassifier(
    iterations=1100,
    learning_rate=0.009,
    depth=6,
    l2_leaf_reg=0.7529,
    random_strength=5.4407,
    bagging_temperature=0.3764,
    random_seed=114,
    verbose=0
)
cb2_model.fit(X_train_fe[CB2_FEATURES], Y_train)

# ④ GradientBoosting
gb_model = GradientBoostingClassifier(
    n_estimators=100,
    learning_rate=0.158,
    max_depth=4,
    min_samples_leaf=19,
    subsample=0.9625,
    max_features=0.3356,
    random_state=42
)
gb_model.fit(X_train_fe[GB_FEATURES], Y_train)

# ⑤ CatBoost
cb3_model = CatBoostClassifier(
    iterations=100,
    learning_rate=0.0053,
    depth=4,
    l2_leaf_reg=20.2913,
    random_strength=6.3387,
    bagging_temperature=0.0338,
    random_seed=42,
    verbose=0
)
cb3_model.fit(X_train_fe[CB3_FEATURES], Y_train)
print("모델 학습 완료")

모델 학습 완료


# 7. 앙상블 예측 (Soft Voting, 5개 모델 확률 동일가중 평균) 및 평가

In [8]:
from sklearn.metrics import (accuracy_score, f1_score, recall_score,
                             precision_score, roc_auc_score)

def ensemble_proba(X):
    p_cb1 = cb1_model.predict_proba(X[CB1_FEATURES])[:, 1]
    p_xgb = xgb_model.predict_proba(X[XGB_FEATURES])[:, 1]
    p_cb2 = cb2_model.predict_proba(X[CB2_FEATURES])[:, 1]
    p_gb  = gb_model.predict_proba(X[GB_FEATURES])[:, 1]
    p_cb3 = cb3_model.predict_proba(X[CB3_FEATURES])[:, 1]
    return (p_cb1 + p_xgb + p_cb2 + p_gb + p_cb3) / 5

# X_valid 예측
Y_valid_prob = ensemble_proba(X_valid_fe)           # AUC용 확률값
Y_valid_pred = (Y_valid_prob >= 0.5).astype(int)

# X_test 예측
Y_test_prob = ensemble_proba(X_test_fe)             # AUC용 확률값
Y_test_pred = (Y_test_prob >= 0.5).astype(int)

# 평가 함수 정의
def evaluate(Y_actual, Y_pred, Y_prob, dataset_name):
    accuracy  = accuracy_score(Y_actual, Y_pred)
    f1        = f1_score(Y_actual, Y_pred)
    recall    = recall_score(Y_actual, Y_pred)
    precision = precision_score(Y_actual, Y_pred)
    auc       = roc_auc_score(Y_actual, Y_prob)

    print(f"\n=== {dataset_name} 평가 ===")
    print(f"Accuracy : {accuracy:.4f}")
    print(f"F1 Score : {f1:.4f}")
    print(f"Recall   : {recall:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"AUC      : {auc:.4f}")

    return {'Accuracy': accuracy, 'F1': f1, 'Recall': recall,
            'Precision': precision, 'AUC': auc}

# Validation, Test 평가 실행
valid_scores = evaluate(Y_valid, Y_valid_pred, Y_valid_prob, "Validation")
test_scores  = evaluate(Y_test,  Y_test_pred,  Y_test_prob,  "Test")


=== Validation 평가 ===
Accuracy : 0.8024
F1 Score : 0.5030
Recall   : 0.3962
Precision: 0.6885
AUC      : 0.9013

=== Test 평가 ===
Accuracy : 0.8786
F1 Score : 0.5143
Recall   : 0.3971
Precision: 0.7297
AUC      : 0.9036


# 8. 최종 결과

In [9]:
import sklearn, xgboost, catboost
print(f"pandas {pd.__version__} / scikit-learn {sklearn.__version__} / xgboost {xgboost.__version__} / catboost {catboost.__version__}")

result = pd.DataFrame({
    'Validation AUC': [valid_scores['AUC']],
    'Test AUC': [test_scores['AUC']],
    '평균 AUC': [(valid_scores['AUC'] + test_scores['AUC']) / 2],
}, index=['5-Model Soft Voting (CatBoost×3 + XGBoost + GB)']).round(5)
result

pandas 3.0.6 / scikit-learn 1.9.1 / xgboost 3.2.0 / catboost 1.2.10


,Validation AUC,Test AUC,평균 AUC
5-Model Soft Voting (CatBoost×3 + XGBoost + GB),0.90127,0.90358,0.90243
